<a href="https://colab.research.google.com/github/rodrigololr/ml/blob/main/Lista_1_Ant%C3%B4nio_Rodrigo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lista 1

**Aluno:** Antônio Rodrigo Lima de Andrade Tenório
\
**Nome da base:** creditcard.csv

## Orientações

- Escolha apenas uma das bases disponíveis e resolva todas as 10 questões usando essa mesma base.
- Desenvolva toda a atividade em Python, no formato de entrega do Google Colab.
- Organize o notebook por questão, com códigos executáveis, saídas geradas e comentários objetivos.
- Não troque de base ao longo da atividade.
- Não faça tratamento manual linha por linha.
- Sempre que necessário, sustente suas decisões com tabelas, métricas, gráficos e resultados do código.

## Importação das bibliotecas

In [1]:
# Bibliotecas principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Pré-processamento e modelagem
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

# Configuração visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## Carregamento da base

In [2]:
# Carregamento da base
# Substitua o nome do arquivo pelo arquivo correto da base escolhida
df = pd.read_csv('creditcard.csv')

# Visualização inicial
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Questão 1 – Diagnóstico estrutural

**Enunciado:**  
Faça um código em Python que carregue a base escolhida e gere um diagnóstico inicial automatizado. Mostre dimensões da base, tipos de dados, valores ausentes, duplicidades, cardinalidade das colunas e possíveis inconsistências de leitura. Em seguida, defina qual será o problema computacional tratado no cenário escolhido, deixando claro se a tarefa será de classificação, regressão ou segmentação. Organize essa etapa em uma função reutilizável.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

- Primeiro passo foi o entendimento da base de dados, pesquisar sobre ela, e enteder sobre o que se trata (Transções fraudulentas)
- Depois disso foi vizualizar o tamanho da tabela, tipo de dadsos, valores ausentes, se tem valores únicos por coluna e possíveis inconstancias

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 1
def gerar_diagnostico(df):

    print(f"dimensões: {df.shape[0]} linhas e {df.shape[1]} colunas.")

    print("\ntipos dos dados:")
    print(df.dtypes)

    print("\nvalores ausentes:")
    print(df.isnull().sum())

    print(f"\nlinhas duplicadas: {df.duplicated().sum()}")

    print("\ncardinalidade (valores únicos por coluna):")
    print(df.nunique())

    print("\npossíveis inconsistências:")
    # Verifica se existem colunas com apenas 1 valor
    colunas_constantes = [col for col in df.columns if df[col].nunique() <= 1]
    if colunas_constantes:
        print(f"- colunas com valores únicos: {colunas_constantes}")
    else:
        print("- nenhuma coluna constante detectada.")

# DEFINIÇÃO DO PROBLEMA:
# O objetivo é analisar transações de cartão de crédito para identificar padrões de uso.
# Vamos utilizar uma tarefa de CLASSIFICAÇÃO para prever se uma transação é fraudulenta ou não.

### Conclusão da Questão 1
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 2 – Auditoria de qualidade

**Enunciado:**  
Faça um código em Python para auditar a qualidade dos dados. Identifique valores implausíveis, colunas com tipos inadequados, categorias raras, padrões suspeitos de preenchimento, possíveis outliers e inconsistências importantes para o cenário escolhido. Se houver datas, verifique coerência temporal. Se houver valores monetários, verifique compatibilidade entre grandezas. Ao final, apresente um resumo dos principais problemas encontrados.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.
- De começo pensei logo em verificar valores que não fazem sentido como:

1- Tempo negativo;
2- Como o tempo está em ordem crescente, verificar se realmente todos estão;
3- Verificar se o "Amout" tem valores negativos

- Verificar se há outliers com **IQR**, mas como os valores são muito próximos decidi pegar apenas o outliers extremos, por isso ao inves de multiplicar por 1.5, vou multiplicar por 3

- Ao analisar a base é nótorio que os casos raros são justamente o que estamos procurando, ou seja, quantidade de Fraudes

### Desenvolvimento
Implemente abaixo o código da questão.

In [45]:
# Código da Questão 2
def detectar_outliers_extremos(df):
    df_num = df.select_dtypes(include=['number']).drop(columns=['Class'], errors='ignore')

    Q1 = df_num.quantile(0.25)
    Q3 = df_num.quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - (3 * IQR)
    limite_superior = Q3 + (3 * IQR)

    valor_outlier = (df_num < limite_inferior) | (df_num > limite_superior)

    total_outliers = valor_outlier.sum()

    return total_outliers[total_outliers > 0]

In [59]:
def auditoria_qualidade(df):

  #incontancias
  inconsistencias_valor = df[df['Amount'] < 0]
  quantidade_negativos = inconsistencias_valor.shape[0]
  checagem_tempo = df['Time'].is_monotonic_increasing
  inconsistencias_tempo = df[df['Time'] < 0]
  quantidade_negativos_tempo = inconsistencias_tempo.shape[0]
  tem_valores_nulos = df.isnull().values.any()

  #outliers
  outliers_totais = detectar_outliers_extremos(df).sum()

  # casos raros
  quantidade_de_fraudes = df[df['Class'] == 1].shape[0]
  quantidade_trancacoes = df.shape[0]
  calculo_fraudes = (quantidade_de_fraudes / quantidade_trancacoes) * 100

  #padrões suspeitos
  padrao_suspeito = df.duplicated(subset=['Time', 'Amount']).sum()

  #tipos inadequados
  colunas_texto = df.select_dtypes(include=['object']).columns.tolist()

  # compatibilidade de grandezas
  media_valor = df['Amount'].mean()
  maximo_valor = df['Amount'].max()
  razao_disparidade = maximo_valor / media_valor


  print("Resumo: ")
  print(f"\n1. VALORES IMPLAUSÍVEIS:  {quantidade_negativos} negativos em Amount/Time.")
  print(f"2. COERÊNCIA TEMPORAL:    OK (Monotônico)")
  print(f"3. VALORES AUSENTES:      OK (0 Nulos)")
  print(f"4. TIPAGEM INADEQUADA:    {len(colunas_texto)} colunas do tipo 'object' encontradas.")
  print(f"5. PADRÕES SUSPEITOS:     {padrao_suspeito} transações com mesmo Tempo/Valor.")
  print(f"6. OUTLIERS EXTREMOS:     {outliers_totais} registros detectados (Critério 3x IQR).")
  print(f"7. COMPATIBILIDADE:       Disparidade de {razao_disparidade:.1f}x (Máximo vs Média).")
  print(f"8. CATEGORIAS RARAS:      {calculo_fraudes:.2f}% de casos de fraude na base.")
  print(f"9. MÉDIA E MÁXIMO:        {media_valor:.2f} e {maximo_valor:.2f}")


auditoria_qualidade(df)

Resumo: 

1. VALORES IMPLAUSÍVEIS:  0 negativos em Amount/Time.
2. COERÊNCIA TEMPORAL:    OK (Monotônico)
3. VALORES AUSENTES:      OK (0 Nulos)
4. TIPAGEM INADEQUADA:    0 colunas do tipo 'object' encontradas.
5. PADRÕES SUSPEITOS:     4863 transações com mesmo Tempo/Valor.
6. OUTLIERS EXTREMOS:     118639 registros detectados (Critério 3x IQR).
7. COMPATIBILIDADE:       Disparidade de 290.8x (Máximo vs Média).
8. CATEGORIAS RARAS:      0.17% de casos de fraude na base.
9. MÉDIA E MÁXIMO:        88.35 e 25691.16


### Conclusão da Questão 2
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 3 – Limpeza dos dados

**Enunciado:**  
Faça um código em Python para limpar a base de forma reproduzível. Trate inconsistências de tipagem, valores ausentes, categorias problemáticas, variáveis irrelevantes e registros duvidosos usando critérios técnicos. Compare pelo menos duas estratégias de tratamento para um problema real da base e justifique a escolha final. Mostre também o impacto quantitativo de cada etapa da limpeza.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 3

### Conclusão da Questão 3
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 4 – Análise exploratória

**Enunciado:**  
Faça um código em Python para desenvolver uma análise exploratória orientada ao problema central da base escolhida. Gere tabelas e visualizações que revelem padrões relevantes entre a variável principal e os demais atributos. Se o cenário envolver classificação, compare os grupos da variável-alvo. Se envolver regressão, investigue relação entre o alvo e as variáveis explicativas. Se envolver segmentação, explore possíveis estruturas de agrupamento. Apresente uma leitura analítica dos resultados obtidos.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 4

### Conclusão da Questão 4
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 5 – Associação entre variáveis

**Enunciado:**  
Faça um código em Python para medir a associação entre as variáveis explicativas e a variável principal da análise. Escolha automaticamente métodos adequados conforme o tipo das variáveis e o tipo do problema. Ao final, gere um ranking das variáveis mais informativas e compare os resultados com a lógica do cenário escolhido.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 5

### Conclusão da Questão 5
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 6 – Engenharia de atributos

**Enunciado:**  
Faça um código em Python para criar pelo menos cinco novos atributos a partir da base original. Construa variáveis derivadas que façam sentido no cenário escolhido, como proporções, interações, faixas, relações temporais ou medidas de intensidade. Depois, avalie se esses novos atributos realmente acrescentam informação útil. Organize essa etapa em uma função reutilizável.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 6

### Conclusão da Questão 6
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 7 – Pré-processamento

**Enunciado:**  
Faça um código em Python para montar um pipeline completo de pré-processamento com `Pipeline` e `ColumnTransformer`. Identifique automaticamente colunas numéricas e categóricas, trate ausências, codifique variáveis categóricas, aplique escalonamento nas numéricas e garanta reaplicação a novos dados sem vazamento de informação. Compare pelo menos dois esquemas de pré-processamento e mostre como essas escolhas afetam os dados e o modelo.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 7

### Conclusão da Questão 7
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 8 – Modelagem com KNN

**Enunciado:**  
Faça um código em Python para dividir a base em treino, validação e teste de forma adequada ao problema escolhido. Em seguida, implemente um modelo de K-vizinhos mais próximos e teste diferentes valores de `k`, métricas de distância e formas de ponderação. Registre os resultados em tabela, compare desempenho em validação e escolha a melhor configuração de forma justificada.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 8

### Conclusão da Questão 8
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 9 – Comparação de modelos

**Enunciado:**  
Faça um código em Python para avaliar o KNN em diferentes cenários de preparação dos dados, como uso ou não de padronização, presença ou ausência de atributos derivados, uso de todas as variáveis ou de um subconjunto selecionado, e tratamento do desbalanceamento quando fizer sentido. Depois, implemente um segundo modelo supervisionado e compare com o melhor KNN em métricas, tempo de execução e perfil dos erros. Conclua se o KNN é ou não uma boa escolha para a base utilizada.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 9

### Conclusão da Questão 9
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 10 – Consolidação da solução

**Enunciado:**  
Faça um código em Python para consolidar toda a solução em um fluxo reutilizável. Implemente uma função que receba um novo caso em formato de dicionário ou `DataFrame` de uma linha, aplique o pipeline construído e retorne a previsão final de forma interpretável. Depois, gere um relatório resumido com a base escolhida, os principais problemas encontrados, os atributos mais relevantes, a melhor configuração do modelo, as métricas finais e uma recomendação executiva. Finalize com uma avaliação crítica sobre a maturidade da solução para uso prático.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 10

### Conclusão da Questão 10
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Conclusão Final

Apresente uma síntese geral do trabalho, destacando:

- principais problemas encontrados na base;
- principais decisões metodológicas;
- melhor configuração de modelo obtida;
- limitações da análise;
- avaliação final da adequação da solução ao cenário escolhido.

**Bom trabalho!**